# 뉴스 상세 한국어 번역 미리 만들기

뉴스 상세 화면(피그마 527:6148)의 **AI 요약 칸(한국어 줄글 한 단락)**과 **영향 한 문장**을 기사가 추출되자마자 미리 만든다(2026-11-20 결정: 누를 때 만들면 처음 여는 사람이 3~8초 기다림).
- 추출 결과의 `comparison_summary`(영어 1~3문장)를 한국어 줄글 한 단락으로 번역한다(2026-11-20 정함). 지시문은 `prompt/ko_translate_prompt.md`.
- 입력: `news_extract`가 쓴 `source/live/articles.jsonl`의 추출 결과 중 제목·`comparison_summary`·회사별 영향만. 기사 본문을 다시 받지 않는다.
- 출력: `source/live/ko_summaries.jsonl`(기사마다 한 줄: key, text, impact_note). `news_to_db`가 `news.article_ko`로 올리고, 화면은 DB만 읽는다.
- 영향 문장은 누구에게나 같은 문장이다(보는 사람의 보유 종목은 화면에서 이름만 붙인다).
- `comparison_summary`가 없는 기사는 만들지 않는다(화면은 '원문 보기' 안내).
- 서버에서는 `run_news.py ko`가 `WATCH_RUN`으로 60초마다 새 추출 결과를 처리한다.


In [ ]:
import datetime as dt, json, os, threading, time, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

ROOT = Path.cwd()
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])
LIVE = ROOT / 'source' / 'live'
ARTICLES_PATH = LIVE / 'articles.jsonl'      # news_extract 결과(읽기만)
OUT_PATH = LIVE / 'ko_summaries.jsonl'       # 기사마다 한 줄: key, text(줄글 번역), impact_note
PROMPT_PATH = ROOT / 'prompt' / 'ko_translate_prompt.md'   # 지시문
USAGE_PATH = LIVE / 'llm_usage.csv'          # LLM 호출마다 토큰 수(다른 노트북과 함께 씀)
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI를 부른다
WATCH_RUN = False        # True면 마지막 칸에서 WATCH_SECONDS초마다 새 추출 결과를 처리
WATCH_SECONDS = 60
WORKERS = 8              # 동시에 보내는 수
KST = dt.timezone(dt.timedelta(hours=9))
LOCK = threading.Lock()


def env_value(name):
    if os.environ.get(name):
        return os.environ[name]
    for line in (PROJ / '.env').read_text(encoding='utf-8').splitlines():
        if line.startswith(name + '='):
            return line.split('=', 1)[1].strip().strip('"').strip("'")
    return None

## 지시문과 호출
지시문은 `prompt/ko_translate_prompt.md`에서 읽는다. 입력은 영어 제목·`comparison_summary`·회사별 영향뿐이다.


In [ ]:
PROMPT = PROMPT_PATH.read_text(encoding='utf-8')


def ask(article):
    body = {'model': LLM_MODEL, 'instructions': PROMPT, 'store': False, 'text': {'format': {'type': 'json_object'}},
            'input': 'json 입력:\n' + json.dumps(article, ensure_ascii=False)}
    req = urllib.request.Request('https://api.openai.com/v1/responses', data=json.dumps(body).encode(), method='POST',
                                 headers={'Authorization': 'Bearer ' + (env_value('OPENAI_API_KEY') or ''), 'Content-Type': 'application/json'})
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                res = json.loads(r.read())
            break
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503) and attempt < 2:
                time.sleep(10 * (attempt + 1))
                continue
            raise RuntimeError(f'OpenAI {e.code}') from None
    text = ''.join(x.get('text', '') for item in res.get('output', []) if item.get('type') == 'message' for x in item.get('content', []))
    return json.loads(text), res.get('usage') or {}


def article_input(r):
    """번역에 넣는 것: 영어 제목(참고), comparison_summary(번역할 글), 회사별 영향(영향 한 문장용)"""
    e = r.get('extraction') or {}
    graph = e.get('graph') or {}
    names = {x.get('id'): x.get('name') for x in graph.get('entities') or []}
    return {'title': r.get('title'), 'comparison_summary': e.get('comparison_summary'),
            'impacts': [{'name': names.get(c.get('entity_id')), 'impact': c.get('impact')} for c in e.get('company_impact') or []]}


## 새 추출 결과 처리
이미 번역한 기사(`ko_summaries.jsonl`에 text가 있는 key)는 건너뛴다. `comparison_summary`가 없는 기사(원문 못 받음·보도자료·영상, 또는 추출에서 비교 요약을 내지 않음)는 만들지 않는다.


In [ ]:
def done_keys():
    """이미 번역한 기사(text가 있는 줄)"""
    if not OUT_PATH.exists():
        return set()
    return {r['key'] for r in map(json.loads, filter(str.strip, OUT_PATH.read_text(encoding='utf-8').splitlines())) if r.get('text')}


def log_usage(key, usage):
    new = not USAGE_PATH.exists()
    with LOCK, USAGE_PATH.open('a', encoding='utf-8') as f:
        if new:
            f.write('time_utc,step,ref,input_tokens,cached_tokens,output_tokens\n')
        f.write(f"{dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')},ko_summary,{key},{usage.get('input_tokens', 0)},"
                f"{(usage.get('input_tokens_details') or {}).get('cached_tokens', 0)},{usage.get('output_tokens', 0)}\n")


def run_once():
    have = done_keys()
    rows = {}
    for l in ARTICLES_PATH.read_text(encoding='utf-8').splitlines():
        if l.strip():
            r = json.loads(l)
            if (r.get('extraction') or {}).get('comparison_summary') and r['key'] not in have:
                rows[r['key']] = r
    if not rows:
        return 0
    if not RUN_LLM:
        print('번역할 기사', len(rows), '(RUN_LLM = False라 부르지 않음)')
        return 0
    made = 0
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(ask, article_input(r)): k for k, r in rows.items()}
        for f in as_completed(futs):
            k = futs[f]
            try:
                out, usage = f.result()
            except Exception as err:
                print('  실패', k, type(err).__name__, str(err)[:120])
                continue
            row = {'key': k, 'text': str(out.get('text') or '').strip(), 'impact_note': str(out.get('impact_note') or ''),
                   'created_kst': dt.datetime.now(KST).strftime('%Y-%m-%d %H:%M:%S'), 'model': LLM_MODEL}
            with LOCK, OUT_PATH.open('a', encoding='utf-8') as fo:
                fo.write(json.dumps(row, ensure_ascii=False) + '\n')
            log_usage(k, usage)
            made += 1
    print(f'[{dt.datetime.now(KST):%m-%d %H:%M}] 한국어 번역 {made}건 만듦(대상 {len(rows)}건)', flush=True)
    return made

## 실행

In [ ]:
if WATCH_RUN:
    print(f'{WATCH_SECONDS}초마다 새 추출 결과를 확인합니다.', flush=True)
    while True:
        try:
            run_once()
        except Exception as e:
            print('오류:', type(e).__name__, str(e)[:200], '(다음에 다시 시도)', flush=True)
        time.sleep(WATCH_SECONDS)
else:
    run_once()